<a href="https://colab.research.google.com/github/arauch6363-crypto/pmu-tracker/blob/main/tracking_auswertung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tracking-Auswertung France Galop
**Bedienung:** Einstellungen prüfen → **Laufzeit → Alle ausführen**.

Jeder Lauf arbeitet **von gestern rückwärts**: erst die neuesten fehlenden Tage, dann weiter in die Vergangenheit bis `START` – höchstens `MAX_TAGE` pro Lauf. Ist die Historie komplett, holt ein Lauf nur noch die neuen Tage.

Außerdem automatisch bei jedem Lauf:
- Hat sich `parse_tracking.py` geändert, werden Tage mit Parser-Fehlern neu ausgewertet (ohne Download).
- Fehlende Starterdaten (PMU war nicht erreichbar) werden nachgeholt.
- Für die letzten `NACHZUEGLER_TAGE` Tage wird nachgeschaut, ob fehlende Tracking-PDFs inzwischen veröffentlicht wurden.

## ⚙️ Einstellungen

In [1]:
START    = '2022-01-01'     # so weit zurück soll die Historie reichen
ENDE     = None             # None = gestern (empfohlen)
MAX_TAGE = 10               # höchstens so viele neue Tage pro Lauf
NACHZUEGLER_TAGE = 3        # fehlende Tracking-PDFs der letzten X Tage bei jedem Lauf erneut abfragen

NUR_FLACH = True            # nur Flachrennen (Hürden/Steeple/Cross werden ignoriert)
NUR_TRACKING_BAHNEN = True  # Starterdaten nur für Bahnen mit Tracking-PDFs
ORDNER = 'PT_ tracking'     # Ordner in „Meine Ablage“

## 1 · Vorbereitung

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install pdfplumber pypdf

import sys, importlib
from pathlib import Path
import pandas as pd

BASE    = Path('/content/drive/MyDrive') / ORDNER
PDF_DIR = BASE / 'pdfs'
PDF_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE))
import parse_tracking, fg_download, tracking_pipeline as tp
for m in (parse_tracking, fg_download, tp):
    importlib.reload(m)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 81.3 MB/s eta 0:00:00


### 🔄 Reset (nur bei Bedarf)
Startet die Historie neu: Fortschritt und alle Parquet-Dateien werden gelöscht. Die PDFs bleiben liegen und werden nur neu ausgewertet (kein erneuter Download); gelernte Bahncodes bleiben ebenfalls.

Zum Ausführen die Raute entfernen, Zelle **einmal** ausführen, Raute wieder setzen.

In [3]:
# tp.reset(BASE)
# tp.reset(BASE, pdfs_loeschen=True)   # zusätzlich alle PDFs löschen (werden neu geladen)

## 2 · Abholen & auswerten (nächste Portion)

In [4]:
bericht = tp.run(START, ENDE, base=BASE, pdf_dir=PDF_DIR, max_tage=MAX_TAGE,
                 nur_tracking=NUR_TRACKING_BAHNEN, nur_flach=NUR_FLACH,
                 nachzuegler_tage=NACHZUEGLER_TAGE)
bericht

2026-09-18: weiterhin ohne Tracking-PDF: 20260918_SAI_R8

Neue Tage: 2026-09-08 rückwärts bis 2026-08-30 (10 Tage)
2026-09-08: 0 neue PDFs
2026-09-08: erledigt · 87 Tracking-Starter, 87 PMU-Starter
2026-09-07: 0 neue PDFs
2026-09-07: erledigt · 90 Tracking-Starter, 90 PMU-Starter
2026-09-06: 0 neue PDFs
2026-09-06: erledigt · 81 Tracking-Starter, 105 PMU-Starter
2026-09-05: 0 neue PDFs
2026-09-05: erledigt · 177 Tracking-Starter, 193 PMU-Starter
2026-09-04: 0 neue PDFs
2026-09-04: erledigt · 141 Tracking-Starter, 141 PMU-Starter
2026-09-03: 0 neue PDFs
2026-09-03: erledigt · 189 Tracking-Starter, 189 PMU-Starter
2026-09-02: 0 neue PDFs
2026-09-02: erledigt · 110 Tracking-Starter, 115 PMU-Starter
2026-09-01: 0 neue PDFs
2026-09-01: erledigt · 92 Tracking-Starter, 92 PMU-Starter
2026-08-31: 8 neue PDFs
2026-08-31: erledigt · 100 Tracking-Starter, 100 PMU-Starter
2026-08-30: 8 neue PDFs
2026-08-30: erledigt · 88 Tracking-Starter, 88 PMU-Starter

Stand 2022-01-01 bis 2026-09-18: offen 1702

,tag,status,tracking_races,tracking_runners,tracking_sections,tracking_obstacles,tracking_leader,fehler,hindernis_uebersprungen,parser,pmu,pmu_races,pmu_runners
0,2026-09-08,erledigt,8,87,688,0,48,0,9,54a61174bb,True,8,87
1,2026-09-07,erledigt,8,90,910,0,48,0,0,54a61174bb,True,8,90
2,2026-09-06,erledigt,9,81,877,0,54,0,0,54a61174bb,True,12,105
3,2026-09-05,erledigt,16,177,1529,0,96,0,0,54a61174bb,True,17,193
4,2026-09-04,erledigt,13,141,1379,0,78,0,2,54a61174bb,True,13,141
5,2026-09-03,erledigt,17,189,1675,0,99,0,0,54a61174bb,True,17,189
6,2026-09-02,erledigt,11,110,1051,0,66,0,4,54a61174bb,True,12,115
7,2026-09-01,erledigt,8,92,667,0,47,0,0,54a61174bb,True,8,92
8,2026-08-31,erledigt,8,100,1006,0,48,0,0,54a61174bb,True,8,100
9,2026-08-30,erledigt,8,88,616,0,48,0,0,54a61174bb,True,8,88


### Manuell neu auswerten (ohne Download)
Normalerweise nicht nötig – Fehler-Tage werden nach einem Parser-Update automatisch neu ausgewertet. Für **alle** Tage (z. B. wenn neue Spalten dazugekommen sind):

In [5]:
# tp.neu_auswerten(BASE, PDF_DIR, nur_fehlertage=False, nur_flach=NUR_FLACH)

## 3 · Fortschritt
Erledigte Tage mit Anzahl Datensätzen. Einen Tag neu abholen: `tp.zuruecksetzen(['2026-09-17'], BASE)` und Zelle 2 erneut ausführen.

In [6]:
st = pd.DataFrame.from_dict(tp.fortschritt(BASE), orient='index')
st.index = pd.to_datetime(st.index, format='%Y%m%d')
offen = tp.offene_tage(START, ENDE, BASE)
fehlt_pmu = int((st.get('pmu') == False).sum()) if 'pmu' in st else 0
print(f'Erledigt: {len(st)} Tage · offen im Zeitraum: {len(offen)} · Starterdaten fehlen: {fehlt_pmu} Tage')
st.sort_index().tail(15)

Erledigt: 20 Tage · offen im Zeitraum: 1702 · Starterdaten fehlen: 0 Tage


,am,fehler,hindernis_uebersprungen,parser,pmu,pmu_races,pmu_runners,tracking_leader,tracking_obstacles,tracking_races,tracking_runners,tracking_sections,fehlende_pdfs
2026-09-04,2026-09-19T12:16:26,0,2,54a61174bb,True,13,141,78,0,13,141,1379,NaN
2026-09-05,2026-09-19T12:16:04,0,0,54a61174bb,True,17,193,96,0,16,177,1529,NaN
2026-09-06,2026-09-19T12:15:38,0,0,54a61174bb,True,12,105,54,0,9,81,877,NaN
2026-09-07,2026-09-19T12:15:22,0,0,54a61174bb,True,8,90,48,0,8,90,910,NaN
2026-09-08,2026-09-19T12:15:10,0,9,54a61174bb,True,8,87,48,0,8,87,688,NaN
2026-09-09,2026-09-19T11:56:31,0,8,54a61174bb,True,8,98,48,0,8,98,822,NaN
2026-09-10,2026-09-19T11:56:03,0,0,54a61174bb,True,9,107,52,0,9,107,908,NaN
2026-09-11,2026-09-19T11:55:45,0,0,54a61174bb,True,16,181,90,0,15,173,1475,NaN
2026-09-12,2026-09-19T11:55:19,0,8,54a61174bb,True,8,90,47,0,8,90,815,NaN
2026-09-13,2026-09-19T11:54:48,0,3,54a61174bb,True,14,174,83,0,14,174,1549,NaN


## 4 · Daten laden
Die Parquet-Dateien liegen in `PT_ tracking/parquet/<tabelle>/<JJJJMMTT>.parquet` – eine Datei pro Tabelle und Tag. Für Databricks/PowerBI kann jeweils der ganze Ordner eingelesen werden.

In [7]:
races    = tp.lade('tracking_races', BASE)
runners  = tp.lade('tracking_runners', BASE)
sections = tp.lade('tracking_sections', BASE)
leader   = tp.lade('tracking_leader', BASE)
pmu_races   = tp.lade('pmu_races', BASE)
pmu_runners = tp.lade('pmu_runners', BASE)
fehler   = tp.lade('fehler', BASE)

print(f'{len(races)} Rennen · {len(runners)} Tracking-Starter · {len(pmu_runners)} PMU-Starter · {len(fehler)} Warnungen')
fehler.tail()

211 Rennen · 2397 Tracking-Starter · 2464 PMU-Starter · 0 Warnungen


""


### Tracking + Starterdaten verbinden

In [8]:
if runners.empty:
    raise RuntimeError('Noch keine Tracking-Daten – Zeitraum prüfen oder Zelle 2 ausführen.')

df = runners.copy()
if not pmu_runners.empty:
    df = df.merge(pmu_runners.drop(columns=['horse']), on=['race_id', 'saddle_no'],
                  how='left', suffixes=('', '_pmu'))
if not pmu_races.empty:
    races = races.merge(pmu_races[['race_id', 'going', 'going_value', 'prize_eur', 'categorie', 'post_time']],
                        on='race_id', how='left', suffixes=('_pdf', ''))

# Wegverlust-bereinigte Zeit
ms = df['avg_speed_kmh'] / 3.6
df['adj_time_s'] = df['official_time_s'] - df['dist_vs_winner_m'] / ms
df['adj_rank'] = df.groupby('race_id')['adj_time_s'].rank(method='min').astype('Int64')
df['place_num'] = pd.to_numeric(df['place'], errors='coerce')
df['rank_gain'] = df['place_num'] - df['adj_rank']
print('mit PMU-Daten:', df['odds_final'].notna().sum() if 'odds_final' in df else 0, 'von', len(df))

mit PMU-Daten: 2363 von 2397


## 11 · Bahnen mit Tracking

In [9]:
codes = pd.read_csv(BASE / 'track_codes.csv')
codes[codes['tracking'] == 'ja'][['hippodrome', 'fg_code']]

,hippodrome,fg_code
0,DIEPPE,DIE
1,CHANTILLY,CHA
2,PARISLONGCHAMP,LON
3,DEAUVILLE,DEA
4,LE LION D ANGERS,LLA
5,NANCY,BRA
6,COMPIEGNE,COM
7,LYON PARILLY,LYO
8,LA CEPIERE,CEP
9,AUTEUIL,AUT
